# Kigurumi Head Reference Agent (v1)

End-to-end Agent that turns a target character + expression reference into:

1. **Step 1** — unobstructed head four-view (general prompt + ≤2 targeted refines).
2. **Dataset pick** — 0–2 visually similar finished shop products from `dataset/`.
3. **Step 2** — kigurumi head shell four-view (general + ≤2 refines).
4. **Step 3** — product-photo four-view (single pass + drift review).

**Models**
- Image: `gpt-image-2` via OpenAI-compatible `images.edit`.
- Reasoning / vision: Qwen3.5-VL (`qwen-vl-max-latest`) via Aliyun Dashscope OpenAI-compatible endpoint.

**Setup** (run from the repo root):
```bash
pip install -r notebook/requirements.txt
cp notebook/.env.example notebook/.env   # then fill in IMAGE_API_KEY and REASON_API_KEY
jupyter notebook kigurumi_agent.ipynb
```

Outputs land in `outputs/<timestamp>_<label>/` — every iteration's candidates plus a `run.json` audit trail.

In [1]:
import sys, importlib
from pathlib import Path

# Allow imports of the agent modules that live under ./notebook/
_module_dir = Path.cwd() / 'notebook'
if str(_module_dir) not in sys.path:
    sys.path.insert(0, str(_module_dir))

import prompts, notebook.config, notebook.agent
importlib.reload(prompts); importlib.reload(notebook.config); importlib.reload(notebook.agent)
from notebook.prompts import load_all
from notebook.config import load_config, DATASET_DIR, OUTPUT_DIR
from notebook.agent import KigurumiAgent

for step_id, sp in load_all().items():
    print(f'step{step_id}: general={len(sp.general)} chars, refines={sp.refinement_keys()}')

cfg = load_config()
print('Image :', cfg.image.model, '@', cfg.image.base_url)
print('Reason:', cfg.reason.model, '@', cfg.reason.base_url)
print('Image API Key:', cfg.image.api_key[:4] + '...' if cfg.image.api_key else None)
print('Reason API Key:', cfg.reason.api_key[:4] + '...' if cfg.reason.api_key else None)
print('Dataset folders:', len(list(DATASET_DIR.iterdir())))
print('Output dir     :', OUTPUT_DIR)
print('image size      :', cfg.image.size)
print('image quality   :', cfg.image.quality)
print('image n         :', cfg.image.n)

step1: general=429 chars, refines=['expression', 'face', 'childlike']
step2: general=802 chars, refines=['shell', 'too3d', 'expression', 'face']
step3: general=474 chars, refines=['four', 'eight']
Image : gpt-image-2-2026-04-21 @ https://api.openai.com/v1
Reason: qwen3.6-plus @ https://dashscope.aliyuncs.com/compatible-mode/v1
Image API Key: sk-p...
Reason API Key: sk-4...
Dataset folders: 82
Output dir     : C:\Project\ai-kigurumi-head-reference\outputs
image size      : 1024x1024
image quality   : high
image n         : 1


## Inputs

All paths below are resolved from the repo root. Drop your reference images into `references/ganyu/` (already gitignored).

- `CHAR_REF_PATHS`: 1–4 official / structure references for the target character.
- `CHAR_INFO`: a short text block: IP, character name, key visual traits — used by the dataset picker and reviewer.
- `EXPR_REF_PATH`: one expression reference image.

In [ ]:
CHAR_REF_PATHS = [
    'references/Niko/front.png',
]
CHAR_INFO = '''蔚蓝档案-妮可，粉色短发带狐狸耳朵的少女，表情是张嘴坏坏的笑'''
EXPR_REF_PATH = 'references/Niko/exp.png'
RUN_LABEL = 'Niko-demo'

In [ ]:
agent_runner = KigurumiAgent(cfg)
result = agent_runner.run(
    char_ref_paths=CHAR_REF_PATHS,
    char_info=CHAR_INFO,
    expr_ref_path=EXPR_REF_PATH,
    run_label=RUN_LABEL,
)
print('Run dir:', result.run_dir)
print('Step 1 best:', result.step1.best.path)
print('Step 2 best:', result.step2.best.path)
print('Step 3 best:', result.step3.best.path)
print('Similar picks:', [p.name for p in result.similar])

## Inspect results inline

In [ ]:
from IPython.display import Image, display, Markdown

def show(label, path):
    display(Markdown(f'### {label}\n`{path}`'))
    display(Image(filename=str(path)))

show('Step 1 — unobstructed head turnaround', result.step1.best.path)
show('Step 2 — kigurumi shell turnaround', result.step2.best.path)
show('Step 3 — product-photo four-view', result.step3.best.path)

if result.similar:
    display(Markdown('### Similar shop products used as references'))
    for p in result.similar:
        show(p.stem, p)

## Audit trail

Every candidate and verdict is saved. `run.json` shows what was generated each iteration, what the reviewer said, and which refine prompt was picked.

In [ ]:
import json
audit = json.loads((result.run_dir / 'run.json').read_text(encoding='utf-8'))
for step in ('step1', 'step2', 'step3'):
    print(f'--- {step} ---')
    for it in audit[step]['iterations']:
        v = it.get('verdict', {})
        print(f"  {it['prompt_key']}: best={v.get('best_index')} verdict={v.get('verdict')} "
              f"refine={v.get('refine_key')} notes={v.get('notes')}")

---

## P0+P1 验证:重跑 hoshino-demo 的 Step 3

上一次跑(`outputs/20260515-194702_hoshino-demo/`)Step 3 跑飞了 —— 把 similar 案例图(圣园未花的双丸子头 + 蓝紫色花饰 + 水印)直接复刻到结果里,完全丢失了 Step 2 的角色身份。

本 cell 验证 P0+P1 改动:
- **P0 photo-role mapping**:把每张 ref 的角色(主参考 / 案例图 / 表情参考)显式拼到 prompt 末尾,告诉模型不同角色的图绝不能互相借用内容
- **P1 drift review**:Step 3 加 `_review_step3_drift`,检查结果是否仍忠于 Step 2 头壳身份;若被案例污染会触发醒目警告

`agent.rerun_step3(source_run_dir)` 复用既有 run 的 step2_best / similar / expr_ref / char_info,只重跑 Step 3,新结果落到独立的 run_dir,不污染原 run。

In [4]:
SOURCE_RUN_DIR = 'outputs/20260515-194702_hoshino-demo'
agent_runner = KigurumiAgent(cfg)
rerun = agent_runner.rerun_step3(
    source_run_dir=SOURCE_RUN_DIR,
    label_suffix='rerun-step3-fixed',
)
print('rerun dir :', rerun.best.path.parent.parent)
print('new best  :', rerun.best.path)
print('drift     :', rerun.iterations[-1]['verdict'])

[20:30:28] [agent] rerun_step3 start: source=20260515-194702_hoshino-demo, new_run_dir=20260515-203028_20260515-194702_hoshino-demo_rerun-step3
[20:30:28] [step3] start: refs=step2_best + 1 similar + expr_ref (no refine loop)
[20:30:28] [step3] generate (key=four)
[20:32:51] [step3] generated 1 candidates in 143.2s
[20:32:51] [step3] drift review
[20:33:46] [step3] drift review (55.3s): best=0, drift=False, fall_back=False, notes="完美保留角色身份：粉色长发、呆毛、异色瞳（左黄右蓝）均与Step 2基准一致。四视图完整，白底商品照风格符合要求。未受案例图（双丸子头、花朵装饰、水印）污染。嘴巴微调为微带笑意，更贴合'带一点点的娇羞'的目标描述，优于Step 2的纯无口表情。"
[20:33:46] [step3] done in 198.5s, best=step3_best.png
[20:33:46] [agent] rerun_step3 complete, run_dir=C:\Project\ai-kigurumi-head-reference\outputs\20260515-203028_20260515-194702_hoshino-demo_rerun-step3
rerun dir : C:\Project\ai-kigurumi-head-reference\outputs\20260515-203028_20260515-194702_hoshino-demo_rerun-step3
new best  : C:\Project\ai-kigurumi-head-reference\outputs\20260515-203028_20260515-194702_hoshino-demo_rerun-step3\step

In [ ]:
from IPython.display import Image, display, Markdown
from pathlib import Path

src = Path(SOURCE_RUN_DIR)
old_step3 = src / 'step3_best.png'
new_step3 = rerun.best.path
step2_best = src / 'step2_best.png'
similar_imgs = sorted((src / 'similar').glob('*'))

display(Markdown('### 基准:Step 2 选中头壳(身份必须保留这一张)'))
display(Image(filename=str(step2_best)))

if similar_imgs:
    display(Markdown('### Similar 案例图(只参考拍摄风格,不能复制角色)'))
    for p in similar_imgs:
        display(Image(filename=str(p)))

display(Markdown('### ❌ 旧 Step 3(跑飞:抄了 similar 的发型/装饰/水印)'))
display(Image(filename=str(old_step3)))

display(Markdown('### ✅ 新 Step 3(P0+P1 修复后)'))
display(Image(filename=str(new_step3)))